In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [2]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
210,NaN,2025-26,1629599,Amir Coffey,Amir,1610612756,PHX,Phoenix Suns,22501185,2026-04-10T00:00:00,PHX @ LAL,L,20.550000,3,6,0.500,1,2,0.500,0,0,0.000,0,2,2,0,1,1,0,0,2,0,7,-11,11.4,0,0,12.0,1,20:33,1,82.4,82.9,82.9,114.2,115.4,115.4,-31.8,-32.5,-32.5,0.000,0.0,0.0,0.000,0.118,0.051,14.3,14.3,0.583,0.583,0.146,0.155,94.22,93.43,77.86,93.43,0.056,41,3.0,6.0,26,77,0.338,7,40,0.175,14,21,0.667,18,29,47,17,23.0,8,1,3,18,18,73,-28.0,80.0,81.1,113.2,112.2,-33.2,-31.1,0.654,0.74,13.2,0.400,0.816,0.570,0.256,0.383,0.423,90.2,90.0,75.00,90,0.255,1610612747,LAL,Los Angeles Lakers,35,69,0.507,8,20,0.400,23,30,0.767,4,31,35,27,11.0,17,3,1,18,18,101,28.0,113.2,112.2,80.0,81.1,33.2,31.1,0.771,2.45,21.8,0.184,0.600,0.430,0.122,0.565,0.614,90.2,90.0,75.00,90,0.745,NaN,SG,28.0
211,NaN,2025-26,1627739,Kris Dunn,Kris,1610612746,LAC,LA Clippers,22501183,2026-04-10T00:00:00,LAC @ POR,L,26.566667,2,3,0.667,1,2,0.500,0,0,0.000,1,4,5,1,2,0,0,0,4,1,5,3,10.5,0,0,12.0,1,26:34,1,128.6,131.4,131.4,117.5,123.1,123.1,11.2,8.3,8.3,0.043,0.5,16.7,0.042,0.143,0.096,33.3,33.3,0.833,0.833,0.088,0.086,96.26,93.05,77.54,93.05,0.031,51,2.0,3.0,36,83,0.434,13,40,0.325,12,12,1.000,10,25,35,17,17.0,6,3,7,27,15,97,-19.0,101.8,102.1,116.7,122.1,-14.9,-20.0,0.472,1.00,13.8,0.250,0.574,0.411,0.179,0.512,0.549,97.3,95.0,79.17,95,0.350,1610612757,POR,Portland Trail Blazers,37,83,0.446,12,39,0.308,30,35,0.857,14,32,46,23,15.0,12,7,3,15,27,116,19.0,116.7,122.1,101.8,102.1,14.9,20.0,0.622,1.53,16.8,0.426,0.750,0.589,0.158,0.518,0.589,97.3,95.0,79.17,95,0.650,G,PG,31.0
212,NaN,2025-26,1641771,Jalen Slawson,Jalen,1610612754,IND,Indiana Pacers,22501175,2026-04-10T00:00:00,IND vs. PHI,L,15.183333,0,4,0.000,0,2,0.000,2,2,1.000,2,3,5,3,5,0,1,0,2,3,2,-14,10.5,0,0,12.0,1,15:11,1,59.0,59.4,59.4,101.9,100.0,100.0,-42.8,-40.6,-40.6,0.500,0.6,23.1,0.111,0.125,0.119,38.5,38.8,0.000,0.205,0.270,0.273,102.11,102.74,85.62,102.74,-0.041,32,0.0,4.0,33,88,0.375,14,50,0.280,14,16,0.875,10,42,52,25,21.0,6,8,2,15,14,94,-11.0,88.6,89.5,100.6,101.0,-12.0,-11.4,0.758,1.19,17.5,0.228,0.723,0.492,0.200,0.455,0.495,105.2,104.5,87.08,105,0.456,1610612755,PHI,Philadelphia 76ers,42,104,0.404,5,29,0.172,16,19,0.842,16,42,58,17,8.0,13,2,8,14,15,105,11.0,100.6,101.0,88.6,89.5,12.0,11.4,0.405,2.13,12.2,0.277,0.772,0.508,0.077,0.428,0.467,105.2,104.5,87.08,104,0.544,NaN,SF,26.0
205,NaN,2025-26,1642362,Payton Sandfort,Payton,161

### Load latest odds on file

In [3]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260412_011456.json


,home_team,away_team,commence_time,bookmakers
0,Miami Heat,Atlanta Hawks,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Boston Celtics,Orlando Magic,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Toronto Raptors,Brooklyn Nets,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,New York Knicks,Charlotte Hornets,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Cleveland Cavaliers,Washington Wizards,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [4]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-12 01:14:56
US latest pull: 2026-04-12 01:13:20


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Bam Adebayo,Over,22.5,-137,2026-04-12,2026-04-12T08:13:54Z,2026-04-12 01:14:56
1,Underdog,player_points,Bam Adebayo,Under,22.5,-137,2026-04-12,2026-04-12T08:13:54Z,2026-04-12 01:14:56
2,Underdog,player_points,Tyler Herro,Over,21.5,-137,2026-04-12,2026-04-12T08:13:54Z,2026-04-12 01:14:56
3,Underdog,player_points,Tyler Herro,Under,21.5,-137,2026-04-12,2026-04-12T08:13:54Z,2026-04-12 01:14:56
4,Underdog,player_points,C.J. McCollum,Over,19.5,-137,2026-04-12,2026-04-12T08:13:54Z,2026-04-12 01:14:56


### Load my models

In [5]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]
min_scaler = min_bundle.get("scaler")

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]
ppm_scaler = ppm_bundle.get("scaler")
#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]
apm_scaler = apm_bundle.get("scaler")

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]
rpm_scaler = rpm_bundle.get("scaler")

### Get Min predictions and Stat Per Min predictions 

In [6]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] Rob Dillingham: single positional indexer is out-of-bounds
[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Lucas Williamson: 'float' object has no attribute 'round'
[SKIP] Toby Okani: 'float' object has no attribute 'round'
[SKIP] Nikola Topic: single positional indexer is out-of-bounds
[SKIP] Branden Carlson: single positional indexer is out-of-bounds
[SKIP] Cormac Ryan: 'float' object has no attribute 'round'
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] Rob Dillingham: single positional indexer is out-of-bounds
[SKIP] Lachlan Olbrich: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: singl

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Andrew Wiggins,AST,14.11,26.88,37.67,0.0004,0.0935,0.1723,0.01,2.51,6.49,"[0.1733102253032928, 0.056657223796034, 0.1289..."
1,Tyler Herro,AST,14.25,31.05,39.83,0.0438,0.1470,0.2442,0.62,4.56,9.73,"[0.1608751608751608, 0.2410929547281007, 0.099..."
2,CJ McCollum,AST,13.62,28.16,37.63,0.0549,0.1699,0.2872,0.75,4.79,10.81,"[0.0741564701520207, 0.0947867298578199, 0.247..."
3,Davion Mitchell,AST,18.00,29.12,37.76,0.0842,0.2067,0.3159,1.52,6.02,11.93,"[0.2074688796680497, 0.18796992481203, 0.15686..."
4,Baylor Scheierman,AST,13.66,21.88,30.27,-0.0007,0.0757,0.1606,-0.01,1.66,4.86,"[0.0811688311688311, 0.0771902740254728, 0.122..."
5,Malachi Smith,AST,26.53,35.10,41.21,0.0174,0.1209,0.2289,0.46,4.24,9.43,"[0.0430663221360895, 0.2222222222222222, 0.0, ..."
6,Ben Saraf,AST,24.53,32.83,38.22,0.0615,0.1960,0.3186,1.51,6.44,12.18,"[0.2814636107760354, 0.2995506739890164, 0.130..."
7,LaMelo Ball,AST,11.80,29.38,37.01,0.1275,0.2620,0.3865,1.50,7.70,14.30,"[0.163291966035271, 0.091324200913242, 0.39239..."
8,Julian Reese,AST,26.72,35.58,40.66,0.0008,0.0951,0.1598,0.02,3.38,6.50,"[0.0, 0.0920810313075506, 0.0520156046814044, ..."
9,Cade Cunningham,AST,12.89,28.80,37.70,0.1373,0.2809,0.4431,1.77,8.09,16.71,"[0.4301786896095301, 0.2871088142405972, 0.340..."


### Get Line Probabilities

In [7]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
88,Franz Wagner,REB,4.5,9.75,22.85,34.24,0.47,3.20,8.33,0.308,0.692
5,Malachi Smith,AST,4.5,26.53,35.10,41.21,0.46,4.24,9.43,0.423,0.577
120,Miles Bridges,PTS,14.5,14.36,26.31,36.68,4.06,13.40,33.24,0.533,0.467
72,Maxime Raynaud,REB,8.5,21.65,32.28,37.86,2.76,8.44,15.96,0.521,0.479
45,Bam Adebayo,REB,11.5,12.92,32.20,40.21,2.01,9.35,17.51,0.306,0.694
17,Jeremiah Fears,AST,4.5,19.49,34.83,42.61,1.20,5.91,11.69,0.571,0.429
150,Kawhi Leonard,PTS,26.5,11.98,30.10,39.65,5.40,22.33,45.58,0.377,0.623
13,Tyus Jones,AST,5.5,13.91,21.52,31.56,-0.00,3.20,10.76,0.268,0.732
153,Bennedict Mathurin,PTS,12.5,11.38,22.44,34.14,2.08,10.92,28.99,0.402,0.599
54,Ausar Thompson,REB,4.5,13.45,24.76,34.72,1.17,5.24,12.05,0.549,0.451


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
47,Jakob Poeltl,REB,6.5,12.44,22.02,33.37,1.73,6.17,15.62,0.370,0.630,REB,Underdog,Brooklyn Nets,-22.0,219.0,117.8,25.0,97.59,27.0,-137.0,-137.0,0.578,0.578,5.3,5.5,2.75,-1.2,-1.0,0.436,0.331,0.669,-42.74,15.73,0.2,0.3,0.40,0.73,23.32,3.88,0.18,0.05,7.00,3.0
136,Collin Sexton,PTS,22.5,15.86,29.58,37.96,5.83,19.42,43.24,0.364,0.636,PTS,Underdog,Dallas Mavericks,6.5,245.5,115.5,19.0,102.52,4.0,-137.0,-137.0,0.578,0.578,19.3,18.5,4.69,-3.2,-4.0,0.682,0.248,0.752,-57.10,30.09,0.2,0.2,0.33,0.15,28.37,3.99,0.25,0.03,15.00,4.0
143,De'Aaron Fox,PTS,18.5,13.78,27.59,38.44,4.93,17.22,40.06,0.423,0.577,PTS,Underdog,Denver Nuggets,-11.0,233.2,116.0,21.0,99.46,20.0,-137.0,-137.0,0.578,0.578,15.0,14.0,5.31,-3.5,-4.5,0.659,0.255,0.745,-55.89,28.88,0.4,0.2,0.27,0.59,28.17,5.72,0.23,0.05,20.40,5.0
3,Davion Mitchell,AST,6.5,18.00,29.12,37.76,1.52,6.02,11.93,0.443,0.557,AST,Underdog,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,6.2,6.0,2.62,-0.3,-0.5,0.115,0.454,0.546,-21.46,-5.55,0.4,0.4,0.27,0.36,30.72,6.67,0.16,0.04,6.12,8.0
76,Toumani Camara,REB,4.5,15.93,30.85,40.47,1.05,4.92,11.71,0.423,0.577,REB,Underdog,Sacramento Kings,-16.5,228.5,120.2,28.0,100.14,17.0,-123.0,-105.0,0.552,0.512,4.4,4.5,1.84,-0.1,0.0,0.054,0.478,0.522,-13.34,1.91,0.4,0.5,0.40,0.60,32.93,5.32,0.18,0.05,5.86,7.0


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
14,Reed Sheppard,AST,4.5,15.08,25.96,35.13,0.29,3.55,8.37,0.316,0.684,AST,PrizePicks,Memphis Grizzlies,-13.0,225.5,118.3,27.0,101.68,8.0,-137.0,-137.0,0.578,0.578,3.2,2.5,2.10,-1.3,-2.0,0.619,0.268,0.732,-53.64,26.63,0.2,0.3,0.33,0.17,25.45,5.76,0.20,0.06,2.00,4.0
0,Andrew Wiggins,AST,2.5,14.11,26.88,37.67,0.01,2.51,6.49,0.446,0.554,AST,PrizePicks,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-130.0,100.0,0.565,0.500,2.1,2.0,1.79,-0.4,-0.5,0.223,0.412,0.588,-27.11,17.60,0.4,0.3,0.40,0.51,27.19,6.71,0.17,0.04,3.17,6.0
192,Rui Hachimura,PTS,14.5,15.23,28.75,38.54,3.65,13.22,31.82,0.415,0.585,PTS,PrizePicks,Utah Jazz,-14.5,236.5,120.8,29.0,103.52,2.0,-137.0,-137.0,0.578,0.578,11.0,12.5,5.72,-3.5,-2.0,0.612,0.270,0.730,-53.29,26.28,0.4,0.2,0.20,0.31,24.66,7.35,0.17,0.05,14.17,6.0
3,Davion Mitchell,AST,6.5,18.00,29.12,37.76,1.52,6.02,11.93,0.443,0.557,AST,PrizePicks,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,6.2,6.0,2.62,-0.3,-0.5,0.115,0.454,0.546,-21.46,-5.55,0.4,0.4,0.27,0.36,30.72,6.67,0.16,0.04,6.12,8.0
20,Miles Bridges,AST,2.5,14.36,26.31,36.68,0.31,2.71,7.41,0.499,0.501,AST,PrizePicks,New York Knicks,-13.0,219.5,112.2,6.0,97.76,25.0,-132.0,102.0,0.569,0.495,2.9,2.5,1.79,0.4,0.0,-0.223,0.588,0.412,3.35,-16.78,0.6,0.5,0.40,0.61,27.97,3.76,0.21,0.06,2.00,4.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
127,Micah Potter,PTS,12.5,14.46,26.27,33.70,2.40,12.12,32.21,0.633,0.366,PTS,Betr DFS,Detroit Pistons,13.5,229.0,108.8,2.0,99.82,19.0,-125.0,-103.0,0.556,0.507,11.9,12.0,6.51,-0.6,-0.5,0.092,0.463,0.537,-16.66,5.84,0.6,0.5,0.33,0.18,20.23,5.01,0.19,0.07,2.00,2.0
178,Jalen Johnson,PTS,22.5,13.23,31.12,39.47,4.24,17.67,39.73,0.266,0.734,PTS,Betr DFS,Miami Heat,4.0,243.0,113.7,13.0,104.22,1.0,-111.0,-116.0,0.526,0.537,20.3,19.0,5.56,-2.2,-3.5,0.396,0.346,0.654,-34.23,21.78,0.0,0.3,0.40,0.42,34.41,5.28,0.26,0.04,24.25,4.0
100,Tyler Herro,PTS,21.5,14.25,31.05,39.83,4.62,18.09,39.64,0.323,0.677,PTS,Betr DFS,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,20.1,18.0,6.67,-1.4,-3.5,0.210,0.417,0.583,-27.86,0.85,0.4,0.3,0.40,0.58,33.59,5.37,0.23,0.04,24.60,5.0
198,Brandon Miller,PTS,19.5,11.18,29.49,38.17,3.47,16.49,38.14,0.403,0.597,PTS,Betr DFS,New York Knicks,-13.0,219.5,112.2,6.0,97.76,25.0,-137.0,-137.0,0.578,0.578,18.9,20.5,6.49,-0.6,1.0,0.092,0.463,0.537,-19.90,-7.10,0.6,0.6,0.60,0.59,31.44,2.36,0.23,0.03,21.25,4.0
181,Ethan Thompson,PTS,11.5,23.69,32.65,38.45,3.96,12.31,29.50,0.607,0.393,PTS,Betr DFS,Detroit Pistons,13.5,229.0,108.8,2.0,99.82,19.0,-137.0,-137.0,0.578,0.578,11.0,13.0,7.44,-0.5,1.5,0.067,0.473,0.527,-18.17,-8.83,0.6,0.5,0.33,0.23,28.95,7.70,0.16,0.05,10.00,1.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
1,Tyler Herro,AST,4.5,14.25,31.05,39.83,0.62,4.56,9.73,0.452,0.548,AST,DraftKings Pick6,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-105.0,-123.0,0.512,0.552,4.4,4.0,1.43,-0.1,-0.5,0.070,0.472,0.528,-7.85,-4.27,0.4,0.4,0.53,0.54,33.59,5.37,0.23,0.04,6.00,5.0
196,Andrew Wiggins,PTS,18.5,14.11,26.88,37.67,3.40,12.86,32.72,0.241,0.759,PTS,DraftKings Pick6,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,13.6,12.5,5.50,-4.9,-6.0,0.891,0.186,0.814,-67.82,40.82,0.4,0.2,0.13,0.35,27.19,6.71,0.17,0.04,18.67,6.0
44,Tyrese Maxey,AST,7.5,14.71,34.07,43.64,1.09,6.04,12.16,0.277,0.723,AST,DraftKings Pick6,Milwaukee Bucks,-15.0,227.0,118.3,26.0,98.29,23.0,118.0,-154.0,0.459,0.606,6.4,7.0,2.50,-1.1,-0.5,0.440,0.330,0.670,-28.06,10.51,0.4,0.5,0.53,0.35,37.62,3.84,0.27,0.04,6.17,6.0
131,Tyrese Maxey,PTS,28.5,14.71,34.07,43.64,5.92,23.66,47.78,0.297,0.703,PTS,DraftKings Pick6,Milwaukee Bucks,-15.0,227.0,118.3,26.0,98.29,23.0,-105.0,-123.0,0.512,0.552,24.8,24.5,4.94,-3.7,-4.0,0.749,0.227,0.773,-55.68,40.15,0.2,0.2,0.33,0.47,37.62,3.84,0.27,0.04,31.50,6.0
99,Bam Adebayo,PTS,23.5,12.92,32.20,40.21,4.16,18.58,39.97,0.216,0.784,PTS,DraftKings Pick6,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,104.0,-135.0,0.490,0.574,18.1,17.5,6.23,-5.4,-6.0,0.867,0.193,0.807,-60.63,40.48,0.4,0.2,0.33,0.25,34.19,5.31,0.23,0.06,16.00,6.0


In [14]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
168,Collin Gillespie,PTS,15.5,18.82,28.06,37.15,3.18,12.61,30.98,0.207,0.793,PTS,Underdog,Oklahoma City Thunder,5.5,213.5,106.1,1.0,100.40,15.0,-137.0,-137.0,0.578,0.578,8.2,10.0,5.35,-7.3,-5.5,1.364,0.086,0.914,-85.12,58.12,0.0,0.1,0.20,0.27,25.06,5.84,0.17,0.04,10.50,6.0
3,Davion Mitchell,AST,6.5,18.00,29.12,37.76,1.52,6.02,11.93,0.443,0.557,AST,Underdog,Atlanta Hawks,-4.0,243.0,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,6.2,6.0,2.62,-0.3,-0.5,0.115,0.454,0.546,-21.46,-5.55,0.4,0.4,0.27,0.36,30.72,6.67,0.16,0.04,6.12,8.0
156,LeBron James,PTS,27.5,12.65,31.94,41.05,5.53,22.34,49.01,0.253,0.747,PTS,Underdog,Utah Jazz,-14.5,236.5,120.8,29.0,103.52,2.0,-137.0,-137.0,0.578,0.578,19.3,17.5,7.10,-8.2,-10.0,1.155,0.124,0.876,-78.55,51.54,0.4,0.2,0.20,0.26,33.73,3.96,0.24,0.06,21.57,7.0
108,Desmond Bane,PTS,19.5,11.90,29.22,37.97,3.97,17.13,35.22,0.379,0.621,PTS,Betr DFS,Boston Celtics,-12.0,217.5,111.8,4.0,95.46,30.0,-137.0,-137.0,0.578,0.578,19.1,19.5,7.50,0.6,1.0,-0.080,0.532,0.468,-7.97,-19.04,0.6,0.5,0.40,0.56,31.14,5.87,0.22,0.05,15.00,5.0
7,LaMelo Ball,AST,7.5,11.80,29.38,37.01,1.50,7.70,14.30,0.428,0.572,AST,Betr DFS,New York Knicks,-13.0,219.5,112.2,6.0,97.76,25.0,-120.0,-108.0,0.545,0.519,7.6,8.0,2.17,0.1,0.5,-0.046,0.518,0.482,-5.03,-7.17,0.8,0.7,0.53,0.46,30.59,4.92,0.30,0.06,6.75,4.0


### Get top EVs for 2 legs

In [15]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 108  |  Pairs: 256  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 88  |  Pairs: 206  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 13  |  Pairs: 10  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 83  |  Pairs: 217  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 108  |  Triples: 5749  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 88  |  Triples: 3820  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 83  |  Triples: 4148  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 13  |  Triples: 29  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
